# Check Summary Statistics for Klebsiella Simulation (New)

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from Bio import Phylo
import seaborn as sns
import torch
from torch.distributions import Uniform
import sbi
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.neural_nets import posterior_nn
from sbi.inference import NPE_C
from sbi.analysis import plot_summary
import sys
sys.path.append('../pysimARG')
from discrete_uniform import DiscreteUniform
from LeaveLengthOut_NN import LeaveLengthOut_NN

torch_device = "cpu"

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load K. pneumoniae gene data and clonal tree.

In [2]:
# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../data/klebsiella/klebsiella_clonal.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

         ___________________________ 6K89HyFBbDwXdwchwqkADP-GCA_019928025
       _|
      | |    _______________________ o3RhFr5btpW7DTfXtue7c5-GCA_019927485
      | |___|
      |     |_______________________ spc6uUrrN2v9jc6UX7DiRy-GCA_019930485
     ,|
     ||       ______________________ w7tJjbUBPZePmFG3NDmzp5-GCA_019928665
     ||  ____|
     || |    |______________________ xaAjEgiWJ6VpYxwkH9LSk9-GCA_019928595
     ||_|
     |  |     ______________________ nSFymoP4AQEsMnKgY82Syr-GCA_019928285
     |  |____|
     |       |______________________ 1k2mmojDYi7HwWwByEje5E-GCA_019928115
     |
     |        ______________________ 5rzNEErwwW9sBnVoSPeLxJ-GCA_019927845
     |   ____|
     |  |    |______________________ 7iUximMjZSb8j3oo5o3GyN-GCA_019927325
  ___|  |
 |   | _|     ______________________ 19UUUUQvAAyHsrWsprUbF7-GCA_019927745
 |   || |____|
 |   || |    |______________________ pVybzfzKoAGacFHHpXyyko-GCA_019928045
 |   || |
 |   || |     ______________________ bKMwQNtdgmF9bNgpmS6F

In [3]:
drop_col = range(16, 32)

In [4]:
x_obs_8000_df1 = pd.read_csv("../data/klebsiella/summary_stats/rand_seg8000_i.csv", header=None)
x_obs_8000_df2 = pd.read_csv("../data/klebsiella/summary_stats/rand_seg8000_ii.csv", header=None)
x_obs_np = np.concatenate((x_obs_8000_df1.to_numpy(), x_obs_8000_df2.to_numpy()), axis=0)
x_obs_np = np.delete(x_obs_np, drop_col, axis=1)
x_obs_torch = torch.tensor(x_obs_np, device=torch_device)
x_obs_torch = x_obs_torch.to(torch.float32)
x_obs_torch.shape, x_obs_torch.dtype

(torch.Size([2000, 30]), torch.float32)

In [5]:
x_obs_rand_df = pd.read_csv("../data/klebsiella/summary_stats/rand_seg_2.csv", header=None)
x_obs_np2 = x_obs_rand_df.to_numpy()
x_obs_np2 = np.delete(x_obs_np2, drop_col, axis=1)
x_obs_torch2 = torch.tensor(x_obs_np2, device=torch_device)
x_obs_torch2 = x_obs_torch2.to(torch.float32)
x_obs_torch2.shape, x_obs_torch2.dtype

(torch.Size([2000, 30]), torch.float32)

### Delete observations with no signal

In [6]:
no_signal_id = np.where(x_obs_np[:, 17] == 0)[0]
no_signal_id.shape, no_signal_id[:10]

((0,), array([], dtype=int64))

In [7]:
x_obs_np = np.delete(x_obs_np, no_signal_id, axis=0)
x_obs_torch = torch.tensor(x_obs_np, device=torch_device)
x_obs_torch = x_obs_torch.to(torch.float32)
x_obs_np.shape, x_obs_torch.shape, x_obs_torch.dtype

((2000, 30), torch.Size([2000, 30]), torch.float32)

In [8]:
no_signal_id2 = np.where(x_obs_np2[:, 17] == 0)[0]
no_signal_id2.shape, no_signal_id2[:10]

((0,), array([], dtype=int64))

In [9]:
x_obs_np2 = np.delete(x_obs_np2, no_signal_id2, axis=0)
x_obs_torch2 = torch.tensor(x_obs_np2, device=torch_device)
x_obs_torch2 = x_obs_torch2.to(torch.float32)
x_obs_np2.shape, x_obs_torch2.shape, x_obs_torch2.dtype

((2000, 30), torch.Size([2000, 30]), torch.float32)

## Load simulation data

In [10]:
theta1 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta1.csv', delimiter=",")
x1 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x1.csv', delimiter=",")
nrow1 = x1.shape[0]

theta2 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta2.csv', delimiter=",")
x2 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x2.csv', delimiter=",")
nrow2 = x2.shape[0]

theta3 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta3.csv', delimiter=",")
x3 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x3.csv', delimiter=",")
nrow3 = x3.shape[0]

theta4 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta4.csv', delimiter=",")
x4 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x4.csv', delimiter=",")
nrow4 = x4.shape[0]

theta5 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta5.csv', delimiter=",")
x5 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x5.csv', delimiter=",")
nrow5 = x5.shape[0]

theta6 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta6.csv', delimiter=",")
x6 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x6.csv', delimiter=",")
nrow6 = x6.shape[0]

theta7 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta7.csv', delimiter=",")
x7 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x7.csv', delimiter=",")
nrow7 = x7.shape[0]

theta8 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta8.csv', delimiter=",")
x8 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x8.csv', delimiter=",")
nrow8 = x8.shape[0]

theta9 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta9.csv', delimiter=",")
x9 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x9.csv', delimiter=",")
nrow9 = x9.shape[0]

theta10 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/theta10.csv', delimiter=",")
x10 = np.loadtxt('../data/klebsiella/ClonalOrigin_sim/x10.csv', delimiter=",")
nrow10 = x10.shape[0]

x = np.vstack([x1, x2, x3, x4, x5, x6, x7, x8, x9, x10])
x = np.delete(x, drop_col, axis=1)
theta = np.vstack([theta1[:nrow1], theta2[:nrow2], theta3[:nrow3], theta4[:nrow4], theta5[:nrow5],
                   theta6[:nrow6], theta7[:nrow7], theta8[:nrow8], theta9[:nrow9], theta10[:nrow10]])

print(theta.shape, x.shape)

(19300, 3) (19300, 30)


In [11]:
theta = torch.tensor(theta, device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = torch.tensor(x, device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

In [12]:
np.min(theta_numpy, axis=0), np.max(theta_numpy, axis=0)

(array([1.7171000e-06, 2.3564592e-06, 1.0100000e+02], dtype=float32),
 array([4.9999490e-02, 4.9998254e-02, 9.9990000e+03], dtype=float32))

### Find out-of-range observations

In [13]:
ignore_indices = []
no_segregation = []
out_stats = dict()
for i in range(x_obs_torch2.shape[0]):
    ignore_i = False
    out_index = []
    for j in range(30):
        max_j = torch.max(x[:, j])
        min_j = torch.min(x[:, j])
        obs_j = x_obs_torch2[i, j]
        if obs_j < min_j or obs_j > max_j:
            ignore_i = True
            out_index.append(j)
        if j == 17 and obs_j == 0:
            no_segregation.append(i)
    if ignore_i or torch.isnan(x_obs_torch2[i, :]).any():
        print(f"Observation {i} is outside the range of simulated data.")
        ignore_indices.append(i)
        out_stats[i] = out_index

Observation 0 is outside the range of simulated data.
Observation 2 is outside the range of simulated data.
Observation 3 is outside the range of simulated data.
Observation 4 is outside the range of simulated data.
Observation 13 is outside the range of simulated data.
Observation 19 is outside the range of simulated data.
Observation 29 is outside the range of simulated data.
Observation 30 is outside the range of simulated data.
Observation 32 is outside the range of simulated data.
Observation 52 is outside the range of simulated data.
Observation 55 is outside the range of simulated data.
Observation 62 is outside the range of simulated data.
Observation 77 is outside the range of simulated data.
Observation 78 is outside the range of simulated data.
Observation 80 is outside the range of simulated data.
Observation 84 is outside the range of simulated data.
Observation 90 is outside the range of simulated data.
Observation 107 is outside the range of simulated data.
Observation 1

In [14]:
len(ignore_indices), len(no_segregation), len(out_stats)

(225, 0, 225)

In [15]:
out_index_all = []
for i in range(len(ignore_indices)):
    idx = ignore_indices[i]
    out_index_all += out_stats[idx]

In [16]:
from collections import Counter

integer_counts = Counter(out_index_all)

In [17]:
for i in range(30):
    print(f"Index {i}: {integer_counts[i]}")

Index 0: 0
Index 1: 0
Index 2: 0
Index 3: 0
Index 4: 0
Index 5: 0
Index 6: 0
Index 7: 0
Index 8: 47
Index 9: 32
Index 10: 0
Index 11: 0
Index 12: 68
Index 13: 88
Index 14: 0
Index 15: 0
Index 16: 0
Index 17: 0
Index 18: 0
Index 19: 0
Index 20: 0
Index 21: 0
Index 22: 137
Index 23: 18
Index 24: 0
Index 25: 124
Index 26: 3
Index 27: 0
Index 28: 3
Index 29: 0


In [18]:
print("Simulation prop of segregating sites:", [np.min(x_numpy[:, 17]), np.max(x_numpy[:, 17])])
print("Observation prop of segregating sites:", [np.min(x_obs_np[:, 17]), np.max(x_obs_np[:, 17])])

Simulation prop of segregating sites: [np.float32(0.0), np.float32(0.9520767)]
Observation prop of segregating sites: [np.float64(0.005875), np.float64(0.139)]


In [19]:
print("Simulation homoplasy index:", [np.min(x_numpy[:, 16]), np.max(x_numpy[:, 16])])
print("Observation homoplasy index:", [np.min(x_obs_np[:, 16]), np.max(x_obs_np[:, 16])])

Simulation homoplasy index: [np.float32(0.0), np.float32(0.9166667)]
Observation homoplasy index: [np.float64(0.4125), np.float64(0.8663074591682101)]


In [20]:
print("Simulation Hudson's Rm:", [np.min(x_numpy[:, 24]), np.max(x_numpy[:, 24])])
print("Observation Hudson's Rm:", [np.min(x_obs_np[:, 24]), np.max(x_obs_np[:, 24])])

Simulation Hudson's Rm: [np.float32(0.0), np.float32(2929.0)]
Observation Hudson's Rm: [np.float64(3.0), np.float64(266.0)]
